# Notebook 06: Silver-to-Gold Aggregations

## Objective

This notebook transforms the validated PaySim Silver dataset into business-ready
Gold tables.

The Gold layer will provide:

- daily transaction summaries;
- hourly fraud-monitoring summaries;
- transaction-type summaries;
- origin-account activity summaries;
- destination-account activity summaries;
- high-value transaction summaries;
- fraud-monitoring datasets;
- an analytics-ready fraud feature table.

The Gold layer is designed for reporting, SQL analytics, dashboards, and
downstream machine-learning consumers.

Balance columns are retained in Silver but excluded from the fraud feature table
because the dataset documentation warns that they may introduce target leakage.

In [1]:
import os
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)
from pyspark.sql.window import Window

In [2]:
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

print("Python executable:", sys.executable)

Python executable: c:\Projects\paysim-financial-data-pipeline\.venv\Scripts\python.exe


In [3]:
spark = (
    SparkSession.builder
    .appName("PaySimSilverToGold")
    .master("local[4]")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Driver memory:", spark.conf.get("spark.driver.memory"))
print(
    "Shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions"),
)

c:\Projects\paysim-financial-data-pipeline\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version: 4.2.0
Driver memory: 8g
Shuffle partitions: 64


In [4]:
PROJECT_ROOT = Path(
    r"C:\Projects\paysim-financial-data-pipeline"
)

RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "PS_20174392719_1491204439457_log.csv"
)

GOLD_SUMMARY_OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "gold"
    / "summary_exports"
)

AUDIT_OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "gold"
    / "pipeline_audit"
)

GOLD_SUMMARY_OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True,
)

AUDIT_OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True,
)

print("Raw source:", RAW_DATA_PATH)
print("Source exists:", RAW_DATA_PATH.exists())

Raw source: C:\Projects\paysim-financial-data-pipeline\data\raw\PS_20174392719_1491204439457_log.csv
Source exists: True


In [5]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Source file not found: {RAW_DATA_PATH}"
    )

In [6]:
transaction_schema = StructType(
    [
        StructField("step", IntegerType(), nullable=True),
        StructField("type", StringType(), nullable=True),
        StructField("amount", DoubleType(), nullable=True),
        StructField("nameOrig", StringType(), nullable=True),
        StructField("oldbalanceOrg", DoubleType(), nullable=True),
        StructField("newbalanceOrig", DoubleType(), nullable=True),
        StructField("nameDest", StringType(), nullable=True),
        StructField("oldbalanceDest", DoubleType(), nullable=True),
        StructField("newbalanceDest", DoubleType(), nullable=True),
        StructField("isFraud", IntegerType(), nullable=True),
        StructField("isFlaggedFraud", IntegerType(), nullable=True),
    ]
)

In [7]:
pipeline_run_id = str(uuid.uuid4())

pipeline_start_timestamp = datetime.now(
    timezone.utc
)

source_filename = RAW_DATA_PATH.name

print("Pipeline run ID:", pipeline_run_id)
print("Pipeline start:", pipeline_start_timestamp)

Pipeline run ID: 8be59cd7-9b19-4889-81c3-7adc20adbd19
Pipeline start: 2026-07-25 23:56:41.365225+00:00


In [8]:
raw_df = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .option("encoding", "UTF-8")
    .option("mode", "PERMISSIVE")
    .schema(transaction_schema)
    .csv(str(RAW_DATA_PATH))
)

In [9]:
SOURCE_COLUMNS = [
    "step",
    "type",
    "amount",
    "nameOrig",
    "oldbalanceOrg",
    "newbalanceOrig",
    "nameDest",
    "oldbalanceDest",
    "newbalanceDest",
    "isFraud",
    "isFlaggedFraud",
]

In [10]:
record_hash_expression = F.sha2(
    F.concat_ws(
        "||",
        *[
            F.coalesce(
                F.col(column).cast("string"),
                F.lit("<NULL>"),
            )
            for column in SOURCE_COLUMNS
        ],
    ),
    256,
)

In [11]:
bronze_df = (
    raw_df
    .withColumn(
        "_pipeline_run_id",
        F.lit(pipeline_run_id),
    )
    .withColumn(
        "_ingestion_timestamp",
        F.current_timestamp(),
    )
    .withColumn(
        "_ingestion_date",
        F.to_date(F.current_timestamp()),
    )
    .withColumn(
        "_source_file",
        F.lit(source_filename),
    )
    .withColumn(
        "_source_file_path",
        F.lit(str(RAW_DATA_PATH)),
    )
    .withColumn(
        "_record_hash",
        record_hash_expression,
    )
)

In [12]:
standardized_df = (
    bronze_df
    .withColumnRenamed(
        "type",
        "transaction_type",
    )
    .withColumnRenamed(
        "nameOrig",
        "origin_account",
    )
    .withColumnRenamed(
        "oldbalanceOrg",
        "old_balance_origin",
    )
    .withColumnRenamed(
        "newbalanceOrig",
        "new_balance_origin",
    )
    .withColumnRenamed(
        "nameDest",
        "destination_account",
    )
    .withColumnRenamed(
        "oldbalanceDest",
        "old_balance_destination",
    )
    .withColumnRenamed(
        "newbalanceDest",
        "new_balance_destination",
    )
    .withColumnRenamed(
        "isFraud",
        "is_fraud",
    )
    .withColumnRenamed(
        "isFlaggedFraud",
        "is_flagged_fraud",
    )
    .withColumn(
        "transaction_type",
        F.upper(F.trim(F.col("transaction_type"))),
    )
    .withColumn(
        "origin_account",
        F.trim(F.col("origin_account")),
    )
    .withColumn(
        "destination_account",
        F.trim(F.col("destination_account")),
    )
)

In [13]:
HIGH_VALUE_THRESHOLD = 200_000.0

silver_enriched_df = (
    standardized_df
    .withColumn(
        "transaction_day",
        F.ceil(
            F.col("step") / F.lit(24)
        ).cast("integer"),
    )
    .withColumn(
        "hour_of_day",
        (
            (F.col("step") - F.lit(1))
            % F.lit(24)
        ).cast("integer"),
    )
    .withColumn(
        "destination_category",
        F.when(
            F.col("destination_account").startswith("M"),
            F.lit("MERCHANT"),
        )
        .when(
            F.col("destination_account").startswith("C"),
            F.lit("CUSTOMER"),
        )
        .otherwise(
            F.lit("UNKNOWN"),
        ),
    )
    .withColumn(
        "is_customer_destination",
        (
            F.col("destination_category")
            == F.lit("CUSTOMER")
        ).cast("integer"),
    )
    .withColumn(
        "is_merchant_destination",
        (
            F.col("destination_category")
            == F.lit("MERCHANT")
        ).cast("integer"),
    )
    .withColumn(
        "is_high_value_transaction",
        (
            F.col("amount")
            > F.lit(HIGH_VALUE_THRESHOLD)
        ).cast("integer"),
    )
    .withColumn(
        "is_zero_amount",
        (
            F.col("amount") == 0
        ).cast("integer"),
    )
    .withColumn(
        "is_self_transfer",
        (
            F.col("origin_account")
            == F.col("destination_account")
        ).cast("integer"),
    )
    .withColumn(
        "amount_category",
        F.when(
            F.col("amount") == 0,
            F.lit("ZERO"),
        )
        .when(
            F.col("amount") < 1_000,
            F.lit("LOW"),
        )
        .when(
            F.col("amount") < 50_000,
            F.lit("MEDIUM"),
        )
        .when(
            F.col("amount") <= 200_000,
            F.lit("HIGH"),
        )
        .otherwise(
            F.lit("VERY_HIGH"),
        ),
    )
)

In [14]:
VALID_TRANSACTION_TYPES = [
    "CASH_IN",
    "CASH_OUT",
    "DEBIT",
    "PAYMENT",
    "TRANSFER",
]

In [15]:
validated_df = (
    silver_enriched_df
    .withColumn(
        "fail_missing_critical_field",
        (
            F.col("step").isNull()
            | F.col("transaction_type").isNull()
            | F.col("amount").isNull()
            | F.col("origin_account").isNull()
            | F.col("destination_account").isNull()
            | F.col("is_fraud").isNull()
            | F.col("is_flagged_fraud").isNull()
        ).cast("integer"),
    )
    .withColumn(
        "fail_invalid_transaction_type",
        (
            ~F.col("transaction_type").isin(
                VALID_TRANSACTION_TYPES
            )
        ).cast("integer"),
    )
    .withColumn(
        "fail_invalid_step",
        (
            ~F.col("step").between(1, 744)
        ).cast("integer"),
    )
    .withColumn(
        "fail_negative_amount",
        (
            F.col("amount") < 0
        ).cast("integer"),
    )
    .withColumn(
        "fail_invalid_fraud_flag",
        (
            ~F.col("is_fraud").isin(0, 1)
        ).cast("integer"),
    )
    .withColumn(
        "fail_invalid_flagged_fraud",
        (
            ~F.col("is_flagged_fraud").isin(0, 1)
        ).cast("integer"),
    )
    .withColumn(
        "fail_invalid_origin_account",
        (
            ~F.col("origin_account").rlike(
                r"^C[0-9]+$"
            )
        ).cast("integer"),
    )
    .withColumn(
        "fail_invalid_destination_account",
        (
            ~F.col("destination_account").rlike(
                r"^[CM][0-9]+$"
            )
        ).cast("integer"),
    )
)

In [16]:
hard_failure_columns = [
    "fail_missing_critical_field",
    "fail_invalid_transaction_type",
    "fail_invalid_step",
    "fail_negative_amount",
    "fail_invalid_fraud_flag",
    "fail_invalid_flagged_fraud",
    "fail_invalid_origin_account",
    "fail_invalid_destination_account",
]

In [17]:
validated_df = validated_df.withColumn(
    "hard_failure_count",
    sum(
        F.col(column)
        for column in hard_failure_columns
    ),
)

In [18]:
silver_final_df = (
    validated_df
    .filter(
        F.col("hard_failure_count") == 0
    )
)

In [19]:
silver_record_count = silver_final_df.count()

print(
    "Silver record count:",
    f"{silver_record_count:,}",
)

Silver record count: 6,362,620


In [20]:
silver_final_df.show(n=10)

+----+----------------+--------+--------------+------------------+------------------+-------------------+-----------------------+-----------------------+--------+----------------+--------------------+--------------------+---------------+--------------------+--------------------+--------------------+---------------+-----------+--------------------+-----------------------+-----------------------+-------------------------+--------------+----------------+---------------+---------------------------+-----------------------------+-----------------+--------------------+-----------------------+--------------------------+---------------------------+--------------------------------+------------------+
|step|transaction_type|  amount|origin_account|old_balance_origin|new_balance_origin|destination_account|old_balance_destination|new_balance_destination|is_fraud|is_flagged_fraud|    _pipeline_run_id|_ingestion_timestamp|_ingestion_date|        _source_file|   _source_file_path|        _record_hash|

In [21]:
silver_final_df.cache()

DataFrame[step: int, transaction_type: string, amount: double, origin_account: string, old_balance_origin: double, new_balance_origin: double, destination_account: string, old_balance_destination: double, new_balance_destination: double, is_fraud: int, is_flagged_fraud: int, _pipeline_run_id: string, _ingestion_timestamp: timestamp, _ingestion_date: date, _source_file: string, _source_file_path: string, _record_hash: string, transaction_day: int, hour_of_day: int, destination_category: string, is_customer_destination: int, is_merchant_destination: int, is_high_value_transaction: int, is_zero_amount: int, is_self_transfer: int, amount_category: string, fail_missing_critical_field: int, fail_invalid_transaction_type: int, fail_invalid_step: int, fail_negative_amount: int, fail_invalid_fraud_flag: int, fail_invalid_flagged_fraud: int, fail_invalid_origin_account: int, fail_invalid_destination_account: int, hard_failure_count: int]

In [22]:
silver_record_count = silver_final_df.count()

print(
    "Cached Silver records:",
    f"{silver_record_count:,}",
)

Cached Silver records: 6,362,620


In [23]:
gold_daily_transaction_summary_df = (
    silver_final_df
    .groupBy(
        "transaction_day",
    )
    .agg(
        F.count("*").alias(
            "transaction_count"
        ),
        F.round(
            F.sum("amount"),
            2,
        ).alias(
            "total_transaction_amount"
        ),
        F.round(
            F.avg("amount"),
            2,
        ).alias(
            "average_transaction_amount"
        ),
        F.round(
            F.min("amount"),
            2,
        ).alias(
            "minimum_transaction_amount"
        ),
        F.round(
            F.max("amount"),
            2,
        ).alias(
            "maximum_transaction_amount"
        ),
        F.sum("is_fraud").alias(
            "fraud_count"
        ),
        F.round(
            F.sum(
                F.when(
                    F.col("is_fraud") == 1,
                    F.col("amount"),
                ).otherwise(0)
            ),
            2,
        ).alias(
            "fraud_amount"
        ),
        F.sum(
            "is_flagged_fraud"
        ).alias(
            "flagged_fraud_count"
        ),
        F.sum(
            "is_high_value_transaction"
        ).alias(
            "high_value_transaction_count"
        ),
        F.countDistinct(
            "origin_account"
        ).alias(
            "unique_origin_accounts"
        ),
        F.countDistinct(
            "destination_account"
        ).alias(
            "unique_destination_accounts"
        ),
    )
    .withColumn(
        "fraud_rate_pct",
        F.round(
            F.col("fraud_count")
            / F.col("transaction_count")
            * F.lit(100),
            6,
        ),
    )
    .withColumn(
        "fraud_amount_pct",
        F.round(
            F.col("fraud_amount")
            / F.col("total_transaction_amount")
            * F.lit(100),
            6,
        ),
    )
    .orderBy(
        "transaction_day"
    )
)
gold_daily_transaction_summary_df.show(
    n=10,
    truncate=False,
)

+---------------+-----------------+------------------------+--------------------------+--------------------------+--------------------------+-----------+--------------+-------------------+----------------------------+----------------------+---------------------------+--------------+----------------+
|transaction_day|transaction_count|total_transaction_amount|average_transaction_amount|minimum_transaction_amount|maximum_transaction_amount|fraud_count|fraud_amount  |flagged_fraud_count|high_value_transaction_count|unique_origin_accounts|unique_destination_accounts|fraud_rate_pct|fraud_amount_pct|
+---------------+-----------------+------------------------+--------------------------+--------------------------+--------------------------+-----------+--------------+-------------------+----------------------------+----------------------+---------------------------+--------------+----------------+
|1              |574255           |9.213186562565E10       |160437.2                  |0.1       

In [24]:
gold_daily_basic_summary_df = (
    silver_final_df
    .groupBy("transaction_day")
    .agg(
        F.count("*").alias("transaction_count"),

        F.round(
            F.sum("amount"),
            2,
        ).alias("total_transaction_amount"),

        F.round(
            F.avg("amount"),
            2,
        ).alias("average_transaction_amount"),

        F.round(
            F.min("amount"),
            2,
        ).alias("minimum_transaction_amount"),

        F.round(
            F.max("amount"),
            2,
        ).alias("maximum_transaction_amount"),

        F.sum("is_fraud").alias("fraud_count"),

        F.round(
            F.sum(
                F.when(
                    F.col("is_fraud") == 1,
                    F.col("amount"),
                ).otherwise(F.lit(0.0))
            ),
            2,
        ).alias("fraud_amount"),

        F.sum("is_flagged_fraud").alias(
            "flagged_fraud_count"
        ),

        F.sum("is_high_value_transaction").alias(
            "high_value_transaction_count"
        ),
    )
    .withColumn(
        "fraud_rate_pct",
        F.when(
            F.col("transaction_count") > 0,
            F.round(
                F.col("fraud_count")
                / F.col("transaction_count")
                * F.lit(100.0),
                6,
            ),
        ).otherwise(F.lit(0.0)),
    )
    .withColumn(
        "fraud_amount_pct",
        F.when(
            F.col("total_transaction_amount") > 0,
            F.round(
                F.col("fraud_amount")
                / F.col("total_transaction_amount")
                * F.lit(100.0),
                6,
            ),
        ).otherwise(F.lit(0.0)),
    )
    .orderBy("transaction_day")
)

gold_daily_basic_summary_df.show(
    n=10,
    truncate=False,
)

+---------------+-----------------+------------------------+--------------------------+--------------------------+--------------------------+-----------+--------------+-------------------+----------------------------+--------------+----------------+
|transaction_day|transaction_count|total_transaction_amount|average_transaction_amount|minimum_transaction_amount|maximum_transaction_amount|fraud_count|fraud_amount  |flagged_fraud_count|high_value_transaction_count|fraud_rate_pct|fraud_amount_pct|
+---------------+-----------------+------------------------+--------------------------+--------------------------+--------------------------+-----------+--------------+-------------------+----------------------------+--------------+----------------+
|1              |574255           |9.213186562565E10       |160437.2                  |0.1                       |1.0E7                     |271        |2.1116382746E8|0                  |155750                      |0.047192      |0.229197        |


In [25]:
gold_daily_account_counts_df = (
    silver_final_df
    .groupBy("transaction_day")
    .agg(
        F.approx_count_distinct(
            "origin_account",
            rsd=0.02,
        ).alias("estimated_unique_origin_accounts"),

        F.approx_count_distinct(
            "destination_account",
            rsd=0.02,
        ).alias("estimated_unique_destination_accounts"),
    )
)

In [26]:
gold_daily_account_counts_df.show(
    n=10,
    truncate=False,
)

+---------------+--------------------------------+-------------------------------------+
|transaction_day|estimated_unique_origin_accounts|estimated_unique_destination_accounts|
+---------------+--------------------------------+-------------------------------------+
|7              |419640                          |277622                               |
|5              |10006                           |10029                                |
|2              |454264                          |237577                               |
|1              |564366                          |247776                               |
|3              |1068                            |1065                                 |
|6              |432541                          |258228                               |
|4              |28226                           |27765                                |
|11             |401981                          |323727                               |
|8              |4417

In [27]:
gold_daily_transaction_summary_df = (
    gold_daily_basic_summary_df
    .join(
        gold_daily_account_counts_df,
        on="transaction_day",
        how="left",
    )
    .orderBy("transaction_day")
)

In [28]:
gold_daily_transaction_summary_df.show(
    n=10,
    truncate=False,
)

+---------------+-----------------+------------------------+--------------------------+--------------------------+--------------------------+-----------+--------------+-------------------+----------------------------+--------------+----------------+--------------------------------+-------------------------------------+
|transaction_day|transaction_count|total_transaction_amount|average_transaction_amount|minimum_transaction_amount|maximum_transaction_amount|fraud_count|fraud_amount  |flagged_fraud_count|high_value_transaction_count|fraud_rate_pct|fraud_amount_pct|estimated_unique_origin_accounts|estimated_unique_destination_accounts|
+---------------+-----------------+------------------------+--------------------------+--------------------------+--------------------------+-----------+--------------+-------------------+----------------------------+--------------+----------------+--------------------------------+-------------------------------------+
|1              |574255           |9.

In [29]:
gold_hourly_fraud_summary_df = (
    silver_final_df
    .groupBy(
        "step",
        "transaction_day",
        "hour_of_day",
    )
    .agg(
        F.count("*").alias(
            "transaction_count"
        ),
        F.round(
            F.sum("amount"),
            2,
        ).alias(
            "total_transaction_amount"
        ),
        F.sum("is_fraud").alias(
            "fraud_count"
        ),
        F.round(
            F.sum(
                F.when(
                    F.col("is_fraud") == 1,
                    F.col("amount"),
                ).otherwise(0)
            ),
            2,
        ).alias(
            "fraud_amount"
        ),
        F.sum(
            "is_flagged_fraud"
        ).alias(
            "flagged_fraud_count"
        ),
        F.sum(
            "is_high_value_transaction"
        ).alias(
            "high_value_transaction_count"
        ),
        F.countDistinct(
            "origin_account"
        ).alias(
            "unique_origin_accounts"
        ),
        F.countDistinct(
            "destination_account"
        ).alias(
            "unique_destination_accounts"
        ),
    )
    .withColumn(
        "fraud_rate_pct",
        F.round(
            F.col("fraud_count")
            / F.col("transaction_count")
            * F.lit(100),
            6,
        ),
    )
    .withColumn(
        "flag_coverage_pct",
        F.when(
            F.col("fraud_count") > 0,
            F.round(
                F.col("flagged_fraud_count")
                / F.col("fraud_count")
                * F.lit(100),
                6,
            ),
        ).otherwise(F.lit(0.0)),
    )
    .orderBy(
        "step"
    )
)

In [30]:
gold_hourly_fraud_summary_df.show(n=10)

+----+---------------+-----------+-----------------+------------------------+-----------+-------------+-------------------+----------------------------+----------------------+---------------------------+--------------+-----------------+
|step|transaction_day|hour_of_day|transaction_count|total_transaction_amount|fraud_count| fraud_amount|flagged_fraud_count|high_value_transaction_count|unique_origin_accounts|unique_destination_accounts|fraud_rate_pct|flag_coverage_pct|
+----+---------------+-----------+-----------------+------------------------+-----------+-------------+-------------------+----------------------------+----------------------+---------------------------+--------------+-----------------+
|   1|              1|          0|             2708|          2.8542918115E8|         16|   3740247.01|                  0|                         451|                  2708|                       1633|      0.590842|              0.0|
|   2|              1|          1|             1014|

In [31]:
gold_transaction_type_summary_df = (
    silver_final_df
    .groupBy(
        "transaction_type"
    )
    .agg(
        F.count("*").alias(
            "transaction_count"
        ),
        F.round(
            F.sum("amount"),
            2,
        ).alias(
            "total_amount"
        ),
        F.round(
            F.avg("amount"),
            2,
        ).alias(
            "average_amount"
        ),
        F.round(
            F.expr(
                "percentile_approx(amount, 0.5)"
            ),
            2,
        ).alias(
            "median_amount"
        ),
        F.round(
            F.min("amount"),
            2,
        ).alias(
            "minimum_amount"
        ),
        F.round(
            F.max("amount"),
            2,
        ).alias(
            "maximum_amount"
        ),
        F.sum("is_fraud").alias(
            "fraud_count"
        ),
        F.round(
            F.sum(
                F.when(
                    F.col("is_fraud") == 1,
                    F.col("amount"),
                ).otherwise(0)
            ),
            2,
        ).alias(
            "fraud_amount"
        ),
        F.sum(
            "is_flagged_fraud"
        ).alias(
            "flagged_fraud_count"
        ),
        F.sum(
            "is_high_value_transaction"
        ).alias(
            "high_value_transaction_count"
        ),
    )
    .withColumn(
        "transaction_pct",
        F.round(
            F.col("transaction_count")
            / F.lit(silver_record_count)
            * F.lit(100),
            4,
        ),
    )
    .withColumn(
        "fraud_rate_pct",
        F.round(
            F.col("fraud_count")
            / F.col("transaction_count")
            * F.lit(100),
            6,
        ),
    )
    .orderBy(
        F.desc("transaction_count")
    )
)

In [32]:
gold_transaction_type_summary_df.show(n=10, truncate=False)

+----------------+-----------------+------------------+--------------+-------------+--------------+--------------+-----------+---------------+-------------------+----------------------------+---------------+--------------+
|transaction_type|transaction_count|total_amount      |average_amount|median_amount|minimum_amount|maximum_amount|fraud_count|fraud_amount   |flagged_fraud_count|high_value_transaction_count|transaction_pct|fraud_rate_pct|
+----------------+-----------------+------------------+--------------+-------------+--------------+--------------+-----------+---------------+-------------------+----------------------------+---------------+--------------+
|CASH_OUT        |2237500          |3.9441299522449E11|176273.96     |147081.69    |0.0           |1.0E7         |4116       |5.98920224383E9|0                  |788559                      |35.1663        |0.183955      |
|PAYMENT         |2151495          |2.809337113837E10 |13057.6       |9482.36      |0.02          |238637.98

In [33]:
gold_origin_account_summary_df = (
    silver_final_df
    .groupBy(
        "origin_account"
    )
    .agg(
        F.count("*").alias(
            "transaction_count"
        ),
        F.round(
            F.sum("amount"),
            2,
        ).alias(
            "total_transaction_amount"
        ),
        F.round(
            F.avg("amount"),
            2,
        ).alias(
            "average_transaction_amount"
        ),
        F.round(
            F.min("amount"),
            2,
        ).alias(
            "minimum_transaction_amount"
        ),
        F.round(
            F.max("amount"),
            2,
        ).alias(
            "maximum_transaction_amount"
        ),
        F.countDistinct(
            "destination_account"
        ).alias(
            "unique_destination_count"
        ),
        F.countDistinct(
            "transaction_type"
        ).alias(
            "unique_transaction_type_count"
        ),
        F.min("step").alias(
            "first_transaction_step"
        ),
        F.max("step").alias(
            "last_transaction_step"
        ),
        F.sum("is_fraud").alias(
            "fraud_transaction_count"
        ),
        F.round(
            F.sum(
                F.when(
                    F.col("is_fraud") == 1,
                    F.col("amount"),
                ).otherwise(0)
            ),
            2,
        ).alias(
            "fraud_transaction_amount"
        ),
        F.sum(
            "is_flagged_fraud"
        ).alias(
            "flagged_fraud_count"
        ),
        F.sum(
            "is_high_value_transaction"
        ).alias(
            "high_value_transaction_count"
        ),
        F.sum(
            F.when(
                F.col("transaction_type") == "TRANSFER",
                1,
            ).otherwise(0)
        ).alias(
            "transfer_count"
        ),
        F.sum(
            F.when(
                F.col("transaction_type") == "CASH_OUT",
                1,
            ).otherwise(0)
        ).alias(
            "cash_out_count"
        ),
        F.sum(
            F.when(
                F.col("transaction_type") == "PAYMENT",
                1,
            ).otherwise(0)
        ).alias(
            "payment_count"
        ),
    )
    .withColumn(
        "fraud_rate_pct",
        F.round(
            F.col("fraud_transaction_count")
            / F.col("transaction_count")
            * F.lit(100),
            6,
        ),
    )
)

In [34]:
gold_origin_account_summary_df.show(n=10)

+--------------+-----------------+------------------------+--------------------------+--------------------------+--------------------------+------------------------+-----------------------------+----------------------+---------------------+-----------------------+------------------------+-------------------+----------------------------+--------------+--------------+-------------+--------------+
|origin_account|transaction_count|total_transaction_amount|average_transaction_amount|minimum_transaction_amount|maximum_transaction_amount|unique_destination_count|unique_transaction_type_count|first_transaction_step|last_transaction_step|fraud_transaction_count|fraud_transaction_amount|flagged_fraud_count|high_value_transaction_count|transfer_count|cash_out_count|payment_count|fraud_rate_pct|
+--------------+-----------------+------------------------+--------------------------+--------------------------+--------------------------+------------------------+-----------------------------+---------

In [35]:
gold_destination_account_summary_df = (
    silver_final_df
    .groupBy(
        "destination_account",
        "destination_category",
    )
    .agg(
        F.count("*").alias(
            "received_transaction_count"
        ),
        F.round(
            F.sum("amount"),
            2,
        ).alias(
            "received_total_amount"
        ),
        F.round(
            F.avg("amount"),
            2,
        ).alias(
            "received_average_amount"
        ),
        F.round(
            F.max("amount"),
            2,
        ).alias(
            "received_maximum_amount"
        ),
        F.countDistinct(
            "origin_account"
        ).alias(
            "unique_origin_count"
        ),
        F.min("step").alias(
            "first_received_step"
        ),
        F.max("step").alias(
            "last_received_step"
        ),
        F.sum("is_fraud").alias(
            "fraud_received_count"
        ),
        F.round(
            F.sum(
                F.when(
                    F.col("is_fraud") == 1,
                    F.col("amount"),
                ).otherwise(0)
            ),
            2,
        ).alias(
            "fraud_received_amount"
        ),
        F.sum(
            "is_flagged_fraud"
        ).alias(
            "flagged_fraud_received_count"
        ),
        F.sum(
            "is_high_value_transaction"
        ).alias(
            "high_value_received_count"
        ),
    )
    .withColumn(
        "fraud_received_rate_pct",
        F.round(
            F.col("fraud_received_count")
            / F.col("received_transaction_count")
            * F.lit(100),
            6,
        ),
    )
)

In [36]:
gold_destination_account_summary_df.filter(
    F.col("fraud_received_count") > 0
).orderBy(
    F.desc("fraud_received_count"),
    F.desc("fraud_received_amount"),
).show(
    n=20,
    truncate=False,
)

+-------------------+--------------------+--------------------------+---------------------+-----------------------+-----------------------+-------------------+-------------------+------------------+--------------------+---------------------+----------------------------+-------------------------+-----------------------+
|destination_account|destination_category|received_transaction_count|received_total_amount|received_average_amount|received_maximum_amount|unique_origin_count|first_received_step|last_received_step|fraud_received_count|fraud_received_amount|flagged_fraud_received_count|high_value_received_count|fraud_received_rate_pct|
+-------------------+--------------------+--------------------------+---------------------+-----------------------+-----------------------+-------------------+-------------------+------------------+--------------------+---------------------+----------------------------+-------------------------+-----------------------+
|C668046170         |CUSTOMER        

In [37]:
gold_high_value_summary_df = (
    silver_final_df
    .filter(
        F.col("is_high_value_transaction") == 1
    )
    .groupBy(
        "transaction_day",
        "transaction_type",
    )
    .agg(
        F.count("*").alias(
            "high_value_transaction_count"
        ),
        F.round(
            F.sum("amount"),
            2,
        ).alias(
            "high_value_total_amount"
        ),
        F.round(
            F.avg("amount"),
            2,
        ).alias(
            "high_value_average_amount"
        ),
        F.round(
            F.max("amount"),
            2,
        ).alias(
            "high_value_maximum_amount"
        ),
        F.sum("is_fraud").alias(
            "high_value_fraud_count"
        ),
        F.sum(
            "is_flagged_fraud"
        ).alias(
            "high_value_flagged_count"
        ),
    )
    .withColumn(
        "high_value_fraud_rate_pct",
        F.round(
            F.col("high_value_fraud_count")
            / F.col("high_value_transaction_count")
            * F.lit(100),
            6,
        ),
    )
    .orderBy(
        "transaction_day",
        "transaction_type",
    )
)

In [38]:
gold_high_value_summary_df.show(
    n=5,
    truncate=False,
)

+---------------+----------------+----------------------------+-----------------------+-------------------------+-------------------------+----------------------+------------------------+-------------------------+
|transaction_day|transaction_type|high_value_transaction_count|high_value_total_amount|high_value_average_amount|high_value_maximum_amount|high_value_fraud_count|high_value_flagged_count|high_value_fraud_rate_pct|
+---------------+----------------+----------------------------+-----------------------+-------------------------+-------------------------+----------------------+------------------------+-------------------------+
|1              |CASH_IN         |42946                       |1.358067710166E10      |316226.82                |1289407.91               |0                     |0                       |0.0                      |
|1              |CASH_OUT        |76731                       |2.522775263348E10      |328781.75                |1.0E7                    |65   

In [39]:
gold_fraud_monitoring_df = (
    silver_final_df
    .filter(
        F.col("is_fraud") == 1
    )
    .select(
        "_record_hash",
        "step",
        "transaction_day",
        "hour_of_day",
        "transaction_type",
        "amount",
        "amount_category",
        "origin_account",
        "destination_account",
        "destination_category",
        "is_high_value_transaction",
        "is_flagged_fraud",
        "_pipeline_run_id",
        "_ingestion_timestamp",
        "_source_file",
    )
)

In [40]:
gold_fraud_monitoring_df.show(
    n=20,
    truncate=False,
)

+----------------------------------------------------------------+----+---------------+-----------+----------------+----------+---------------+--------------+-------------------+--------------------+-------------------------+----------------+------------------------------------+--------------------------+------------------------------------+
|_record_hash                                                    |step|transaction_day|hour_of_day|transaction_type|amount    |amount_category|origin_account|destination_account|destination_category|is_high_value_transaction|is_flagged_fraud|_pipeline_run_id                    |_ingestion_timestamp      |_source_file                        |
+----------------------------------------------------------------+----+---------------+-----------+----------------+----------+---------------+--------------+-------------------+--------------------+-------------------------+----------------+------------------------------------+--------------------------+------

In [41]:
gold_fraud_record_count = (
    gold_fraud_monitoring_df.count()
)

print(
    "Fraud monitoring records:",
    f"{gold_fraud_record_count:,}",
)

Fraud monitoring records: 8,213


In [42]:
origin_history_window = (
    Window
    .partitionBy("origin_account")
    .orderBy(
        F.col("step"),
        F.col("_record_hash"),
    )
    .rowsBetween(
        Window.unboundedPreceding,
        -1,
    )
)

In [43]:
feature_enriched_df = (
    silver_final_df
    .withColumn(
        "prior_origin_transaction_count",
        F.count("*").over(
            origin_history_window
        ),
    )
    .withColumn(
        "prior_origin_total_amount",
        F.sum("amount").over(
            origin_history_window
        ),
    )
    .withColumn(
        "prior_origin_average_amount",
        F.avg("amount").over(
            origin_history_window
        ),
    )
    .withColumn(
        "prior_origin_maximum_amount",
        F.max("amount").over(
            origin_history_window
        ),
    )
    .withColumn(
        "prior_origin_fraud_count",
        F.sum("is_fraud").over(
            origin_history_window
        ),
    )
)

In [44]:
feature_enriched_df = (
    feature_enriched_df
    .fillna(
        {
            "prior_origin_transaction_count": 0,
            "prior_origin_total_amount": 0.0,
            "prior_origin_average_amount": 0.0,
            "prior_origin_maximum_amount": 0.0,
            "prior_origin_fraud_count": 0,
        }
    )
)

In [45]:
previous_transaction_window = (
    Window
    .partitionBy("origin_account")
    .orderBy(
        F.col("step"),
        F.col("_record_hash"),
    )
)

In [46]:
feature_enriched_df = (
    feature_enriched_df
    .withColumn(
        "previous_transaction_step",
        F.lag("step").over(
            previous_transaction_window
        ),
    )
    .withColumn(
        "previous_transaction_amount",
        F.lag("amount").over(
            previous_transaction_window
        ),
    )
    .withColumn(
        "steps_since_previous_transaction",
        F.col("step")
        - F.col("previous_transaction_step"),
    )
)

In [47]:
feature_enriched_df = (
    feature_enriched_df
    .fillna(
        {
            "previous_transaction_amount": 0.0,
            "steps_since_previous_transaction": -1,
        }
    )
)

In [48]:
feature_enriched_df = (
    feature_enriched_df
    .withColumn(
        "is_cash_in",
        (
            F.col("transaction_type") == "CASH_IN"
        ).cast("integer"),
    )
    .withColumn(
        "is_cash_out",
        (
            F.col("transaction_type") == "CASH_OUT"
        ).cast("integer"),
    )
    .withColumn(
        "is_debit",
        (
            F.col("transaction_type") == "DEBIT"
        ).cast("integer"),
    )
    .withColumn(
        "is_payment",
        (
            F.col("transaction_type") == "PAYMENT"
        ).cast("integer"),
    )
    .withColumn(
        "is_transfer",
        (
            F.col("transaction_type") == "TRANSFER"
        ).cast("integer"),
    )
)

In [49]:
gold_fraud_feature_df = (
    feature_enriched_df
    .select(
        "_record_hash",
        "step",
        "transaction_day",
        "hour_of_day",
        "transaction_type",
        "amount",
        "amount_category",
        "origin_account",
        "destination_account",
        "destination_category",
        "is_customer_destination",
        "is_merchant_destination",
        "is_high_value_transaction",
        "is_cash_in",
        "is_cash_out",
        "is_debit",
        "is_payment",
        "is_transfer",
        "prior_origin_transaction_count",
        F.round(
            "prior_origin_total_amount",
            2,
        ).alias(
            "prior_origin_total_amount"
        ),
        F.round(
            "prior_origin_average_amount",
            2,
        ).alias(
            "prior_origin_average_amount"
        ),
        F.round(
            "prior_origin_maximum_amount",
            2,
        ).alias(
            "prior_origin_maximum_amount"
        ),
        "prior_origin_fraud_count",
        "previous_transaction_step",
        "previous_transaction_amount",
        "steps_since_previous_transaction",
        "is_flagged_fraud",
        "is_fraud",
        "_pipeline_run_id",
        "_ingestion_timestamp",
        "_source_file",
    )
)

In [50]:
prohibited_balance_columns = {
    "old_balance_origin",
    "new_balance_origin",
    "old_balance_destination",
    "new_balance_destination",
}

feature_columns = set(
    gold_fraud_feature_df.columns
)

unexpected_balance_columns = (
    feature_columns
    & prohibited_balance_columns
)

print(
    "Balance columns found in feature table:",
    unexpected_balance_columns,
)

Balance columns found in feature table: set()


In [51]:
assert not unexpected_balance_columns, (
    "Balance fields must not be included "
    "in the fraud feature table."
)

print(
    "Balance-field exclusion validation passed."
)

Balance-field exclusion validation passed.


In [52]:
gold_fraud_feature_df.select(
    "_record_hash",
    "step",
    "transaction_type",
    "amount",
    "is_high_value_transaction",
    "prior_origin_transaction_count",
    "prior_origin_average_amount",
    "previous_transaction_amount",
    "steps_since_previous_transaction",
    "is_fraud",
).show(
    n=20,
    truncate=False,
)

+----------------------------------------------------------------+----+----------------+---------+-------------------------+------------------------------+---------------------------+---------------------------+--------------------------------+--------+
|_record_hash                                                    |step|transaction_type|amount   |is_high_value_transaction|prior_origin_transaction_count|prior_origin_average_amount|previous_transaction_amount|steps_since_previous_transaction|is_fraud|
+----------------------------------------------------------------+----+----------------+---------+-------------------------+------------------------------+---------------------------+---------------------------+--------------------------------+--------+
|8a11360320e80d55872d167e9d8eb93d60bc73870a772f324c248fd1cf869508|254 |CASH_IN         |49360.77 |0                        |0                             |0.0                        |0.0                        |-1                         

In [53]:
type_summary_record_total = (
    gold_transaction_type_summary_df
    .agg(
        F.sum(
            "transaction_count"
        ).alias("record_count")
    )
    .first()["record_count"]
)

print(
    "Silver records:",
    f"{silver_record_count:,}",
)

print(
    "Transaction-type Gold total:",
    f"{type_summary_record_total:,}",
)

Silver records: 6,362,620
Transaction-type Gold total: 6,362,620


In [54]:
assert (
    silver_record_count
    == type_summary_record_total
)

print(
    "Transaction-type reconciliation passed."
)

Transaction-type reconciliation passed.


In [55]:
daily_summary_record_total = (
    gold_daily_transaction_summary_df
    .agg(
        F.sum(
            "transaction_count"
        ).alias("record_count")
    )
    .first()["record_count"]
)

assert (
    silver_record_count
    == daily_summary_record_total
)

print(
    "Daily-summary reconciliation passed."
)

Daily-summary reconciliation passed.


In [56]:
hourly_summary_record_total = (
    gold_hourly_fraud_summary_df
    .agg(
        F.sum(
            "transaction_count"
        ).alias("record_count")
    )
    .first()["record_count"]
)

assert (
    silver_record_count
    == hourly_summary_record_total
)

print(
    "Hourly-summary reconciliation passed."
)

Hourly-summary reconciliation passed.


In [57]:
silver_fraud_count = (
    silver_final_df
    .agg(
        F.sum("is_fraud").alias(
            "fraud_count"
        )
    )
    .first()["fraud_count"]
)

gold_fraud_count = (
    gold_fraud_monitoring_df.count()
)

print(
    "Silver fraud count:",
    f"{silver_fraud_count:,}",
)

print(
    "Gold fraud-monitoring count:",
    f"{gold_fraud_count:,}",
)

Silver fraud count: 8,213
Gold fraud-monitoring count: 8,213


In [58]:
assert (
    silver_fraud_count
    == gold_fraud_count
)

print(
    "Fraud-monitoring reconciliation passed."
)

Fraud-monitoring reconciliation passed.


In [59]:
feature_table_count = (
    gold_fraud_feature_df.count()
)

print(
    "Silver count:",
    f"{silver_record_count:,}",
)

print(
    "Feature table count:",
    f"{feature_table_count:,}",
)

Silver count: 6,362,620
Feature table count: 6,362,620


In [60]:
assert (
    silver_record_count
    == feature_table_count
)

print(
    "Fraud-feature count reconciliation passed."
)

Fraud-feature count reconciliation passed.


In [61]:
gold_daily_transaction_summary_df.createOrReplaceTempView(
    "gold_daily_transaction_summary"
)

gold_hourly_fraud_summary_df.createOrReplaceTempView(
    "gold_hourly_fraud_summary"
)

gold_transaction_type_summary_df.createOrReplaceTempView(
    "gold_transaction_type_summary"
)

gold_origin_account_summary_df.createOrReplaceTempView(
    "gold_origin_account_summary"
)

gold_destination_account_summary_df.createOrReplaceTempView(
    "gold_destination_account_summary"
)

gold_fraud_monitoring_df.createOrReplaceTempView(
    "gold_fraud_monitoring"
)

gold_fraud_feature_df.createOrReplaceTempView(
    "gold_fraud_features"
)

In [62]:
spark.sql(
    """
    SELECT
        transaction_type,
        transaction_count,
        total_amount,
        fraud_count,
        fraud_amount,
        fraud_rate_pct
    FROM gold_transaction_type_summary
    ORDER BY fraud_rate_pct DESC
    """
).show(
    truncate=False
)

+----------------+-----------------+------------------+-----------+---------------+--------------+
|transaction_type|transaction_count|total_amount      |fraud_count|fraud_amount   |fraud_rate_pct|
+----------------+-----------------+------------------+-----------+---------------+--------------+
|TRANSFER        |532909           |4.8529198726317E11|4097       |6.06721318401E9|0.768799      |
|CASH_OUT        |2237500          |3.9441299522449E11|4116       |5.98920224383E9|0.183955      |
|DEBIT           |41432            |2.2719922128E8    |0          |0.0            |0.0           |
|CASH_IN         |1399284          |2.3636739191246E11|0          |0.0            |0.0           |
|PAYMENT         |2151495          |2.809337113837E10 |0          |0.0            |0.0           |
+----------------+-----------------+------------------+-----------+---------------+--------------+



In [63]:
gold_daily_type_summary_df = (
    silver_final_df
    .groupBy(
        "transaction_day",
        "transaction_type",
    )
    .agg(
        F.count("*").alias(
            "transaction_count"
        ),
        F.round(
            F.sum("amount"),
            2,
        ).alias(
            "total_amount"
        ),
        F.round(
            F.avg("amount"),
            2,
        ).alias(
            "average_amount"
        ),
        F.sum("is_fraud").alias(
            "fraud_count"
        ),
        F.round(
            F.sum(
                F.when(
                    F.col("is_fraud") == 1,
                    F.col("amount"),
                ).otherwise(0)
            ),
            2,
        ).alias(
            "fraud_amount"
        ),
        F.sum(
            "is_flagged_fraud"
        ).alias(
            "flagged_fraud_count"
        ),
    )
    .withColumn(
        "fraud_rate_pct",
        F.round(
            F.col("fraud_count")
            / F.col("transaction_count")
            * F.lit(100),
            6,
        ),
    )
    .orderBy(
        "transaction_day",
        "transaction_type",
    )
)
gold_daily_type_summary_df.show(
    n=30,
    truncate=False,
)

+---------------+----------------+-----------------+-----------------+--------------+-----------+--------------+-------------------+--------------+
|transaction_day|transaction_type|transaction_count|total_amount     |average_amount|fraud_count|fraud_amount  |flagged_fraud_count|fraud_rate_pct|
+---------------+----------------+-----------------+-----------------+--------------+-----------+--------------+-------------------+--------------+
|1              |CASH_IN         |124026           |2.12381194331E10 |171239.25     |0          |0.0           |0                  |0.0           |
|1              |CASH_OUT        |204397           |3.749451502343E10|183439.65     |141        |1.05052284E8  |0                  |0.068983      |
|1              |DEBIT           |4470             |2.79225835E7     |6246.66       |0          |0.0           |0                  |0.0           |
|1              |PAYMENT         |194676           |2.19243002602E9  |11261.94      |0          |0.0           |

In [64]:
daily_summary_pdf = (
    gold_daily_transaction_summary_df
    .toPandas()
)

daily_type_summary_pdf = (
    gold_daily_type_summary_df
    .toPandas()
)

hourly_fraud_summary_pdf = (
    gold_hourly_fraud_summary_df
    .toPandas()
)

transaction_type_summary_pdf = (
    gold_transaction_type_summary_df
    .toPandas()
)

high_value_summary_pdf = (
    gold_high_value_summary_df
    .toPandas()
)

c:\Projects\paysim-financial-data-pipeline\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Projects\paysim-financial-data-pipeline\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Projects\paysim-financial-data-pipeline\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Projects\paysim-financial-data-pipeline\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning

In [65]:
daily_summary_file = (
    GOLD_SUMMARY_OUTPUT_PATH
    / f"daily_transaction_summary_{pipeline_run_id}.csv"
)

daily_type_summary_file = (
    GOLD_SUMMARY_OUTPUT_PATH
    / f"daily_type_summary_{pipeline_run_id}.csv"
)

hourly_summary_file = (
    GOLD_SUMMARY_OUTPUT_PATH
    / f"hourly_fraud_summary_{pipeline_run_id}.csv"
)

transaction_type_file = (
    GOLD_SUMMARY_OUTPUT_PATH
    / f"transaction_type_summary_{pipeline_run_id}.csv"
)

high_value_file = (
    GOLD_SUMMARY_OUTPUT_PATH
    / f"high_value_summary_{pipeline_run_id}.csv"
)

In [66]:
daily_summary_pdf.to_csv(
    daily_summary_file,
    index=False,
)

daily_type_summary_pdf.to_csv(
    daily_type_summary_file,
    index=False,
)

hourly_fraud_summary_pdf.to_csv(
    hourly_summary_file,
    index=False,
)

transaction_type_summary_pdf.to_csv(
    transaction_type_file,
    index=False,
)

high_value_summary_pdf.to_csv(
    high_value_file,
    index=False,
)

In [67]:
print("Daily summary:", daily_summary_file)
print("Daily type summary:", daily_type_summary_file)
print("Hourly summary:", hourly_summary_file)
print("Transaction type summary:", transaction_type_file)
print("High-value summary:", high_value_file)

Daily summary: C:\Projects\paysim-financial-data-pipeline\data\gold\summary_exports\daily_transaction_summary_8be59cd7-9b19-4889-81c3-7adc20adbd19.csv
Daily type summary: C:\Projects\paysim-financial-data-pipeline\data\gold\summary_exports\daily_type_summary_8be59cd7-9b19-4889-81c3-7adc20adbd19.csv
Hourly summary: C:\Projects\paysim-financial-data-pipeline\data\gold\summary_exports\hourly_fraud_summary_8be59cd7-9b19-4889-81c3-7adc20adbd19.csv
Transaction type summary: C:\Projects\paysim-financial-data-pipeline\data\gold\summary_exports\transaction_type_summary_8be59cd7-9b19-4889-81c3-7adc20adbd19.csv
High-value summary: C:\Projects\paysim-financial-data-pipeline\data\gold\summary_exports\high_value_summary_8be59cd7-9b19-4889-81c3-7adc20adbd19.csv


In [68]:
gold_table_registry = [
    {
        "table_name": "gold_daily_transaction_summary",
        "grain": "one row per transaction day",
        "purpose": "Daily transaction and fraud monitoring",
        "expected_size": "small",
    },
    {
        "table_name": "gold_daily_type_summary",
        "grain": "one row per day and transaction type",
        "purpose": "Daily transaction-type analytics",
        "expected_size": "small",
    },
    {
        "table_name": "gold_hourly_fraud_summary",
        "grain": "one row per PaySim step",
        "purpose": "Hourly fraud monitoring",
        "expected_size": "small",
    },
    {
        "table_name": "gold_transaction_type_summary",
        "grain": "one row per transaction type",
        "purpose": "Transaction-type performance",
        "expected_size": "very small",
    },
    {
        "table_name": "gold_origin_account_summary",
        "grain": "one row per origin account",
        "purpose": "Origin-customer activity analysis",
        "expected_size": "large",
    },
    {
        "table_name": "gold_destination_account_summary",
        "grain": "one row per destination account",
        "purpose": "Destination-account activity analysis",
        "expected_size": "large",
    },
    {
        "table_name": "gold_high_value_summary",
        "grain": "one row per day and transaction type",
        "purpose": "High-value transaction monitoring",
        "expected_size": "small",
    },
    {
        "table_name": "gold_fraud_monitoring",
        "grain": "one row per fraudulent transaction",
        "purpose": "Detailed fraud investigation",
        "expected_size": "medium",
    },
    {
        "table_name": "gold_fraud_features",
        "grain": "one row per transaction",
        "purpose": "Fraud analytics and model input",
        "expected_size": "large",
    },
]

In [69]:
gold_table_registry_pdf = pd.DataFrame(
    gold_table_registry
)

gold_table_registry_pdf

,table_name,grain,purpose,expected_size
0,gold_daily_transaction_summary,one row per transaction day,Daily transaction and fraud monitoring,small
1,gold_daily_type_summary,one row per day and transaction type,Daily transaction-type analytics,small
2,gold_hourly_fraud_summary,one row per PaySim step,Hourly fraud monitoring,small
3,gold_transaction_type_summary,one row per transaction type,Transaction-type performance,very small
4,gold_origin_account_summary,one row per origin account,Origin-customer activity analysis,large
5,gold_destination_account_summary,one row per destination account,Destination-account activity analysis,large
6,gold_high_value_summary,one row per day and transaction type,High-value transaction monitoring,small
7,gold_fraud_monitoring,one row per fraudulent transaction,Detailed fraud investigation,medium
8,gold_fraud_features,one row per transaction,Fraud analytics and model input,large


In [70]:
gold_registry_file = (
    GOLD_SUMMARY_OUTPUT_PATH
    / "gold_table_registry.csv"
)

gold_table_registry_pdf.to_csv(
    gold_registry_file,
    index=False,
)

print(
    "Gold registry written to:",
    gold_registry_file,
)

Gold registry written to: C:\Projects\paysim-financial-data-pipeline\data\gold\summary_exports\gold_table_registry.csv


In [71]:
daily_row_count = (
    gold_daily_transaction_summary_df.count()
)

daily_type_row_count = (
    gold_daily_type_summary_df.count()
)

hourly_row_count = (
    gold_hourly_fraud_summary_df.count()
)

type_summary_row_count = (
    gold_transaction_type_summary_df.count()
)

origin_summary_row_count = (
    gold_origin_account_summary_df.count()
)

destination_summary_row_count = (
    gold_destination_account_summary_df.count()
)

high_value_summary_row_count = (
    gold_high_value_summary_df.count()
)

fraud_monitoring_row_count = (
    gold_fraud_monitoring_df.count()
)

In [72]:
print(daily_row_count)
print(daily_type_row_count)
print(hourly_row_count)
print(type_summary_row_count)
print(origin_summary_row_count)
print(destination_summary_row_count)
print(high_value_summary_row_count)
print(fraud_monitoring_row_count)

31
152
743
5
6353307
2722362
103
8213


In [73]:
pipeline_end_timestamp = datetime.now(
    timezone.utc
)

duration_seconds = (
    pipeline_end_timestamp
    - pipeline_start_timestamp
).total_seconds()

In [74]:
audit_record = {
    "pipeline_run_id": pipeline_run_id,
    "pipeline_name": "paysim_silver_to_gold",
    "source_file": source_filename,
    "pipeline_start_timestamp": (
        pipeline_start_timestamp.isoformat()
    ),
    "pipeline_end_timestamp": (
        pipeline_end_timestamp.isoformat()
    ),
    "duration_seconds": duration_seconds,
    "silver_record_count": silver_record_count,
    "daily_summary_row_count": daily_row_count,
    "daily_type_summary_row_count": daily_type_row_count,
    "hourly_summary_row_count": hourly_row_count,
    "transaction_type_summary_row_count": (
        type_summary_row_count
    ),
    "origin_account_summary_row_count": (
        origin_summary_row_count
    ),
    "destination_account_summary_row_count": (
        destination_summary_row_count
    ),
    "high_value_summary_row_count": (
        high_value_summary_row_count
    ),
    "fraud_monitoring_row_count": (
        fraud_monitoring_row_count
    ),
    "fraud_feature_row_count": (
        feature_table_count
    ),
    "daily_reconciliation_passed": (
        silver_record_count
        == daily_summary_record_total
    ),
    "hourly_reconciliation_passed": (
        silver_record_count
        == hourly_summary_record_total
    ),
    "type_reconciliation_passed": (
        silver_record_count
        == type_summary_record_total
    ),
    "fraud_reconciliation_passed": (
        silver_fraud_count
        == gold_fraud_count
    ),
    "pipeline_status": "SUCCESS",
}

In [75]:
audit_pdf = pd.DataFrame(
    [audit_record]
)

audit_pdf.T

,0
pipeline_run_id,8be59cd7-9b19-4889-81c3-7adc20adbd19
pipeline_name,paysim_silver_to_gold
source_file,PS_20174392719_1491204439457_log.csv
pipeline_start_timestamp,2026-07-25T23:56:41.365225+00:00
pipeline_end_timestamp,2026-07-25T23:59:25.923660+00:00
duration_seconds,164.558435
silver_record_count,6362620
daily_summary_row_count,31
daily_type_summary_row_count,152
hourly_summary_row_count,743


In [76]:
audit_file_path = (
    AUDIT_OUTPUT_PATH
    / f"silver_to_gold_audit_{pipeline_run_id}.csv"
)

audit_pdf.to_csv(
    audit_file_path,
    index=False,
)

print(
    "Gold audit written to:",
    audit_file_path,
)

Gold audit written to: C:\Projects\paysim-financial-data-pipeline\data\gold\pipeline_audit\silver_to_gold_audit_8be59cd7-9b19-4889-81c3-7adc20adbd19.csv


## Final Findings

### Gold tables created

The Silver transaction dataset was transformed into multiple analytics-ready
Gold tables:

- daily transaction summary;
- daily transaction-type summary;
- hourly fraud summary;
- transaction-type summary;
- origin-account activity summary;
- destination-account activity summary;
- high-value transaction summary;
- fraud-monitoring table;
- fraud feature table.

### Business value

The Gold layer supports:

- daily and hourly transaction monitoring;
- fraud trend analysis;
- high-value transfer monitoring;
- account-level activity profiling;
- transaction-type performance reporting;
- dashboard development;
- SQL reporting;
- downstream machine-learning workflows.

### Reconciliation

- Daily Gold transaction totals matched Silver.
- Hourly Gold transaction totals matched Silver.
- Transaction-type Gold totals matched Silver.
- Fraud-monitoring records matched the Silver fraud count.
- The fraud feature table preserved one row per valid Silver transaction.

### Feature-table design

- Balance columns were intentionally excluded.
- Historical account features were computed using Spark window functions.
- Transaction-type indicators were created.
- Previous-transaction features were derived.
- The label `is_fraud` was retained for downstream supervised learning.

### Persistence

Small Gold summaries were exported to CSV using Pandas.

Large Gold tables remain Spark DataFrames because native Windows Spark Parquet
persistence is currently deferred. These tables will later be written through
WSL2, Docker, or another Linux-compatible Spark environment.

## Engineering Decisions

1. Gold tables have clearly defined grains.

2. Each table serves a specific analytical consumer.

3. Aggregate tables are reconciled against the Silver source.

4. Detailed fraud records remain separate from aggregate monitoring tables.

5. The fraud feature table excludes balance columns to avoid documented target
   leakage.

6. Window functions use only prior origin-account records where possible.

7. Small Gold outputs may be exported through Pandas.

8. Large Gold datasets must remain in Spark and should later be persisted as
   Parquet or loaded into PostgreSQL.

9. Account-level Gold tables are not converted to Pandas because of their high
   cardinality.

10. Pipeline audit records document row counts, execution time, and
    reconciliation status.

In [77]:
sorted(
    path.name
    for path in GOLD_SUMMARY_OUTPUT_PATH.iterdir()
)

['daily_transaction_summary_8be59cd7-9b19-4889-81c3-7adc20adbd19.csv',
 'daily_type_summary_8be59cd7-9b19-4889-81c3-7adc20adbd19.csv',
 'gold_table_registry.csv',
 'high_value_summary_8be59cd7-9b19-4889-81c3-7adc20adbd19.csv',
 'hourly_fraud_summary_8be59cd7-9b19-4889-81c3-7adc20adbd19.csv',
 'transaction_type_summary_8be59cd7-9b19-4889-81c3-7adc20adbd19.csv']

In [78]:
silver_final_df.unpersist()

print("Silver DataFrame cache released.")

Silver DataFrame cache released.


In [79]:
spark.stop()

print("Spark session stopped.")

Spark session stopped.
